# Token Trajectory Analysis

Tracks how token representations evolve through layers: velocity,
acceleration, convergence, divergence between tokens, and curvature.

**References:**
- Brunner et al. (2020) "On Identifiability in Transformers"
- Ethayarajh (2019) "How Contextual are Contextualized Word Representations?"

## Why This Matters

Token trajectory tracks how individual token representations evolve through the network. Understanding token-level dynamics reveals how context is integrated, when predictions form, and how different positions interact to produce the output.

**Key references:**
- [Elhage et al. (2021) "A Mathematical Framework for Transformer Circuits"](https://transformer-circuits.pub/2021/framework/index.html) — Foundational framework for attention head and residual stream analysis

In [ ]:
import jax
import jax.numpy as jnp
import numpy as np

from irtk import HookedTransformer, HookedTransformerConfig
from irtk.token_trajectory import (
    token_representation_trajectory,
    trajectory_velocity,
    trajectory_acceleration,
    token_convergence,
    trajectory_curvature,
)

cfg = HookedTransformerConfig(
    n_layers=4, d_model=32, n_ctx=64, d_head=8, n_heads=4, d_vocab=100,
)
model = HookedTransformer(cfg)
key = jax.random.PRNGKey(42)
leaves, treedef = jax.tree.flatten(model)
new_leaves = []
for leaf in leaves:
    if isinstance(leaf, jnp.ndarray) and leaf.dtype == jnp.float32:
        key, subkey = jax.random.split(key)
        new_leaves.append(jax.random.normal(subkey, leaf.shape) * 0.1)
    else:
        new_leaves.append(leaf)
model = jax.tree.unflatten(treedef, new_leaves)

tokens = jnp.array([0, 5, 10, 15, 20, 25, 30, 35, 40, 45])
print(f"Model: {cfg.n_layers} layers, d_model={cfg.d_model}, seq_len={len(tokens)}")

In [ ]:
# 1. Token representation trajectory
traj = token_representation_trajectory(model, tokens, positions=[0, 4, -1])
print(f"Tracked positions: {traj['positions_tracked']}")
for pi, pos in enumerate(traj['positions_tracked']):
    norms_str = ' -> '.join(f"{traj['norms'][pi, l]:.3f}" for l in range(cfg.n_layers + 1))
    print(f"  pos {pos}: norms = {norms_str}")

In [ ]:
# 2. Trajectory velocity
vel = trajectory_velocity(model, tokens)
print(f"Fastest layer: {vel['fastest_layer']}")
print(f"Mean velocity per layer: {vel['mean_velocity'].tolist()}")
for pos in range(len(tokens)):
    v_str = ' '.join(f"{vel['velocities'][pos, l]:.3f}" for l in range(cfg.n_layers))
    print(f"  pos {pos}: {v_str}")

In [ ]:
# 3. Trajectory acceleration
acc = trajectory_acceleration(model, tokens)
print(f"Decelerating: {acc['is_decelerating']}")
print(f"Mean acceleration: {acc['mean_acceleration'].tolist()}")

In [ ]:
# 4. Token convergence between pairs
for a, b in [(0, -1), (0, 1), (4, 5)]:
    conv = token_convergence(model, tokens, pos_a=a, pos_b=b)
    label = 'CONVERGING' if conv['is_converging'] else 'diverging'
    dists = ' -> '.join(f"{d:.3f}" for d in conv['distances'])
    print(f"Pos {a} vs {b}: {label} (rate={conv['convergence_rate']:+.4f})")
    print(f"  distances: {dists}")
    cos_str = ' -> '.join(f"{c:.3f}" for c in conv['cosine_similarities'])
    print(f"  cosines: {cos_str}")

In [ ]:
# 5. Trajectory curvature
curv = trajectory_curvature(model, tokens)
print(f"Straightest position: {curv['straightest_position']}")
print(f"Most curved position: {curv['most_curved_position']}")
print(f"Mean curvature per layer transition: {curv['mean_curvature'].tolist()}")
for pos in range(min(5, len(tokens))):
    c_str = ' '.join(f"{curv['curvatures'][pos, l]:.3f}" for l in range(cfg.n_layers - 1))
    print(f"  pos {pos}: {c_str}")